# CNP watermark robustness — paper Figures 3–4

Reproduces the watermark experiments from **"Concept-Aware Pruning via Disentangled Subspaces for Robust
Convolutional Networks"** (XAI4CV @ CVPR 2026): VGG-16 is pruned to 80% of its filters in 5% steps with
**CNP** (Concept-aware Network Pruning, which ablates the watermark concept subspace) and with **vanilla LRP pruning**.
The notebook then plots, as mean ± std over seeds:

- **top:** overall accuracy on the balanced test set (all four class × watermark subgroups)
- **bottom:** accuracy on the worst-performing, out-of-distribution subgroup: negative-class images with a watermark (**c0w1**)

**Runtime:** use a GPU runtime (*Runtime → Change runtime type*). Each (pruner, seed) run is independent,
writes its own log, and is skipped if its results already exist. If the runtime disconnects, re-run the cells.
With `USE_DRIVE = True`, results persist in Google Drive.

> In the code, `ncp` is the identifier for CNP (the method's earlier name), e.g. `--pruner ncp`.

In [ ]:
# ---- Configuration ----
REPO_URL = "https://github.com/KirinDanek/NCP.v2.git"
BRANCH = "release-cleanup"      # branch to clone (switch to "main" once merged)

EXPERIMENTS = ["carton"]        # "carton" = carton vs. dugong (Fig. 3); add "crate" for crate vs. packet (Fig. 4)
PRUNERS = ["ncp", "vanilla"]    # "ncp" = CNP
SEEDS = [0, 1, 2, 3, 4]         # paper: 5 seeds
TOTAL_PR = 0.80                 # paper: 0.80 (16 pruning steps of 5%); use e.g. 0.40 for a quicker run
SAVE_MODELS = False             # also save pruned checkpoints ({pruner}.pt)

USE_DRIVE = True                # store results in Google Drive (survives runtime resets)
DRIVE_RESULTS = "/content/drive/MyDrive/cnp_watermark_results"

## 1. Environment

In [ ]:
import os, subprocess, sys, time
from pathlib import Path

!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("WARNING: no GPU. Switch to a GPU runtime; full runs are impractical on CPU.")

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RESULTS_DIR = Path(DRIVE_RESULTS)
else:
    RESULTS_DIR = Path("/content/cnp_watermark_results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print("results ->", RESULTS_DIR)

In [ ]:
REPO_DIR = Path("/content/NCP.v2")
if not REPO_DIR.exists():
    !git clone --depth 1 --branch {BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git log -1 --format="commit %h  %s"
!pip install -q -e ".[data]"

# make the freshly installed package importable in this kernel
sys.path.insert(0, str(REPO_DIR / "src"))
import ncp
print("ncp", ncp.__version__)

## 2. Data

Downloads the ImageNet synsets from image-net.org (use is subject to the ImageNet terms of access).
Images are kept on the local Colab disk for fast loading.

Caveat: the paper's crate/packet runs used ILSVRC-style filenames, so a fresh crate/packet download may not
reproduce the exact split.

In [ ]:
TASKS = {"carton": "carton_dugong", "crate": "crate_packet"}
for exp in EXPERIMENTS:
    root = REPO_DIR / "data" / "images" / TASKS[exp] / "original"
    counts = {d.name: len(list(d.iterdir())) for d in root.glob("n*")} if root.exists() else {}
    if len(counts) == 2 and min(counts.values()) > 800:
        print(f"[{exp}] already downloaded: {counts}")
        continue
    !python data/images/download_binary_dataset.py --task {TASKS[exp]}
    counts = {d.name: len(list(d.iterdir())) for d in root.glob("n*")}
    print(f"[{exp}] {counts}")

## 3. Run pruning

Same configuration as `scripts/reproduce_watermark.sh` (paper Table 1): 15-epoch head warmup, LRP-α1β0
ranking on 500 positive-class images, 5% pruning steps, 2 fine-tuning epochs per step, 5 final fine-tuning
epochs, and evaluation on the held-out test split at every step.

Subspace indices are 0-indexed: carton ablates paper subspace 4 → `3`; crate ablates paper subspace 2 → `1`.

In [ ]:
SPURIOUS_SUBSPACE = {"carton": 3, "crate": 1}


def run(exp, pruner, seed):
    run_dir = RESULTS_DIR / exp / pruner / f"seed{seed}"
    if (run_dir / "stats.pt").exists():
        print(f"[skip] {exp} {pruner} seed{seed}: results exist")
        return
    log = RESULTS_DIR / "logs" / f"{exp}_{pruner}_seed{seed}.log"
    log.parent.mkdir(parents=True, exist_ok=True)
    cmd = [
        sys.executable, "experiments/run_watermark_pruning_experiment.py",
        "--experiment", exp, "--pruner", pruner, "--seed", str(seed),
        "--spurious_subspace", str(SPURIOUS_SUBSPACE[exp]),
        "--pr_step", "0.05", "--total_pr", str(TOTAL_PR),
        "--lr", "1e-4", "--momentum", "0.9",
        "--train_batch_size", "32", "--test_batch_size", "32",
        "--warmup_epochs", "15", "--iter_finetune_epochs", "2", "--final_finetune_epochs", "5",
        "--rank_loader_type", "positive_only", "--eval_on_test",
        "--out_dir", str(RESULTS_DIR),
    ] + (["--save_model"] if SAVE_MODELS else [])
    print(f"[run]  {exp} {pruner} seed{seed}  (log: {log})", flush=True)
    t0 = time.time()
    with open(log, "w") as f:
        proc = subprocess.run(cmd, cwd=REPO_DIR, stdout=f, stderr=subprocess.STDOUT)
    if proc.returncode != 0:
        print("".join(open(log).readlines()[-30:]))
        raise RuntimeError(f"run failed: {exp} {pruner} seed{seed}")
    print(f"[done] {exp} {pruner} seed{seed} in {(time.time() - t0) / 60:.1f} min", flush=True)


for exp in EXPERIMENTS:
    for seed in SEEDS:            # interleave pruners so partial results already allow a comparison
        for pruner in PRUNERS:
            run(exp, pruner, seed)

## 4. Figures

Uses `analysis/plot_watermark_results.py`, the same plotting code as the paper figures.
The extra panel plots the minimum accuracy over all four subgroups, to check that c0w1 is indeed the worst.

In [ ]:
import importlib.util
import math
import matplotlib.pyplot as plt
from IPython.display import Image, display

spec = importlib.util.spec_from_file_location("plot_watermark_results", REPO_DIR / "analysis" / "plot_watermark_results.py")
pw = importlib.util.module_from_spec(spec)
spec.loader.exec_module(pw)
%matplotlib inline

N_PRUNE_ITERS = int(round(TOTAL_PR / 0.05))
pw.MAX_PRUNE_ITER = N_PRUNE_ITERS   # plot pruning steps only (the final fine-tune is iteration N+1)
FIG_DIR = RESULTS_DIR / "figures"
SUBGROUPS = {"c0w0": "acc_g0y0", "c0w1": "acc_g1y0", "c1w0": "acc_g0y1", "c1w1": "acc_g1y1"}

loaded = {}
for exp in EXPERIMENTS:
    ncp_ds = pw._load_seeds(RESULTS_DIR, exp, "ncp", SEEDS)
    van_ds = pw._load_seeds(RESULTS_DIR, exp, "vanilla", SEEDS)
    loaded[exp] = (ncp_ds, van_ds)
    n = max(len(ncp_ds), len(van_ds))
    if n == 0:
        print(f"[{exp}] no results yet")
        continue
    not_test = [d["seed"] for d in ncp_ds + van_ds if not d.get("eval_on_test")]
    assert not not_test, f"runs not evaluated on the test split: seeds {not_test}"
    print(f"[{exp}] CNP: {len(ncp_ds)} seeds, vanilla: {len(van_ds)} seeds")

    pw.plot_accuracy(exp, ncp_ds, van_ds, n, str(FIG_DIR / f"{exp}_accuracy.png"))
    pw.plot_c0w1_combined(exp, ncp_ds, van_ds, n, str(FIG_DIR / f"{exp}_c0w1_combined.png"))
    display(Image(str(FIG_DIR / f"{exp}_accuracy.png")))
    display(Image(str(FIG_DIR / f"{exp}_c0w1_combined.png")))

    # check: minimum over all four subgroups
    fig, ax = plt.subplots(figsize=(8, 4))
    for label, color, ds in [("CNP", "steelblue", ncp_ds), ("Vanilla", "crimson", van_ds)]:
        if not ds:
            continue
        per_seed = []
        for d in ds:
            accs = [pw._sg_metric(d, key) for key in SUBGROUPS.values()]
            per_seed.append({it: 100 * min(a[it] for a in accs) for it in accs[0]})
        niters, means, stds = pw._agg(per_seed)
        pw._plot_ms(ax, pw._niters_to_pct(niters), means, stds, color=color, ls="-", label=label)
    ax.set_title(f"{exp.capitalize()} – minimum subgroup accuracy")
    pw._style_ax(ax, ylabel="Accuracy (%)")
    ax.set_ylim(0, 105); ax.grid(True, alpha=0.3); ax.legend()
    plt.show()

In [ ]:
# Summary table: mean ± std at the start (0%) and at the final pruning step, plus which subgroup is worst
import numpy as np

def mean_std(values):
    values = [v for v in values if not math.isnan(v)]
    return f"{np.mean(values):5.1f} ± {np.std(values):4.1f}" if values else "   n/a     "

for exp, (ncp_ds, van_ds) in loaded.items():
    print(f"\n{exp}  (accuracy %, mean ± std over seeds)")
    print(f"{'pruner':8s} {'pruned':>6s}  {'overall':>13s}  {'c0w1':>13s}  worst subgroup (mean)")
    for label, ds in [("CNP", ncp_ds), ("vanilla", van_ds)]:
        if not ds:
            continue
        for it in (0, N_PRUNE_ITERS):
            overall = [pw._acc_metric(d).get(it, float("nan")) for d in ds]
            c0w1 = [100 * pw._sg_metric(d, "acc_g1y0").get(it, float("nan")) for d in ds]
            sub_means = {name: np.nanmean([pw._sg_metric(d, key).get(it, float("nan")) for d in ds])
                         for name, key in SUBGROUPS.items()}
            worst = min(sub_means, key=sub_means.get)
            print(f"{label:8s} {int(round(it * 5)):5d}%  {mean_std(overall)}  {mean_std(c0w1)}  "
                  f"{worst} ({100 * sub_means[worst]:.1f})")

## 5. (Optional) Download results

In [ ]:
import shutil
archive = shutil.make_archive("/content/cnp_watermark_results", "zip", RESULTS_DIR)
print("archive:", archive)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    pass